# Sentiment upgrade: train DistilBERT on EduRABSA
**How to use:** press the ▶ play button on each cell, one at a time, top to bottom. Wait for a cell to finish (the play button stops spinning) before pressing the next.

Before you start: menu **Runtime → Change runtime type → T4 GPU → Save**.

## Step 1. Check the fast card is on

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("NO GPU! Go to Runtime > Change runtime type > T4 GPU, then run this cell again.")

GPU available: True
Tesla T4


## Step 2. Install the tools (takes about 1 minute)

In [ ]:
!pip install -q transformers accelerate

## Step 3. Upload 2 files from your laptop
When you press play, a **Choose files** button appears. Select these two files (hold Ctrl to pick both):
- `data/processed/unified_edurabsa_train.csv`  (training data)
- `data/processed/validation_aspects.csv`  (only used to check the score, never for learning)

In [5]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

Saving unified_edurabsa_train.csv to unified_edurabsa_train.csv
Saving validation_aspects.csv to validation_aspects (1).csv
['unified_edurabsa_train.csv', 'validation_aspects (1).csv']


In [7]:
!mv -f "validation_aspects (1).csv" validation_aspects.csv
import pandas as pd
print(len(pd.read_csv("validation_aspects.csv")))

149


## Step 4. Prepare the data

In [8]:
import pandas as pd, numpy as np

LABELS = ["positive", "neutral", "negative"]
L2I = {l: i for i, l in enumerate(LABELS)}

train = pd.read_csv("unified_edurabsa_train.csv")
train = train[train["sentiment"].isin(LABELS)].copy()
train["aspect_term"] = train["aspect_term"].fillna("general").astype(str)
train["text"] = train["text"].astype(str)

val = pd.read_csv("validation_aspects.csv")
val = val[val["sentiment"].isin(LABELS)].copy()
val["aspect_term"] = val["aspect_term"].fillna("general").astype(str)
val["text"] = val["text"].astype(str)

print("training rows:", len(train))
print(train["sentiment"].value_counts().to_dict())
print("validation rows (campus comments):", len(val))

training rows: 16500
{'positive': 9581, 'negative': 5206, 'neutral': 1713}
validation rows (campus comments): 149


## Step 5. Load DistilBERT and turn words into numbers

In [9]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)

class DS(torch.utils.data.Dataset):
    # the model reads the pair:  [aspect term]  +  [sentence]
    def __init__(self, df):
        self.enc = tok(list(df["aspect_term"]), list(df["text"]),
                       truncation=True, max_length=128, padding=True)
        self.y = [L2I[s] for s in df["sentiment"]]
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.y[i])
        return item

train_ds, val_ds = DS(train), DS(val)
print("ready")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ready


## Step 6. Train (about 10 to 20 minutes)
You will see a table of numbers going down. That is normal: the "loss" number getting smaller means it is learning. **Keep this tab open.**

In [10]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="out",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    save_strategy="no",
    logging_steps=100,
    report_to="none",
    seed=42,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds)
trainer.train()

Step,Training Loss
100,0.776789
200,0.639211
300,0.580152
400,0.545857
500,0.524147
600,0.474856
700,0.464147
800,0.458942
900,0.452350
1000,0.449134


TrainOutput(global_step=1032, training_loss=0.5348982432091883, metrics={'train_runtime': 93.9991, 'train_samples_per_second': 351.067, 'train_steps_per_second': 10.979, 'total_flos': 1092875528448000.0, 'train_loss': 0.5348982432091883, 'epoch': 2.0})

## Step 7. Check the score on your campus validation comments
Compare with the baseline (**0.687 macro-F1**). This is campus text the model has never seen, so it is a domain-transfer result: trained on university course/staff reviews, tested on campus facilities.

In [11]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

pred = trainer.predict(val_ds).predictions.argmax(axis=1)
gold = np.array(val_ds.y)
print("macro-F1:", round(f1_score(gold, pred, average="macro", labels=[0,1,2], zero_division=0), 3))
print("accuracy:", round(accuracy_score(gold, pred), 3))
print(confusion_matrix(gold, pred, labels=[0,1,2]), " rows=gold, cols=pred", LABELS)
print(classification_report(gold, pred, labels=[0,1,2], target_names=LABELS, zero_division=0))

macro-F1: 0.708
accuracy: 0.772
[[45  2 13]
 [ 4  8  6]
 [ 7  2 62]]  rows=gold, cols=pred ['positive', 'neutral', 'negative']
              precision    recall  f1-score   support

    positive       0.80      0.75      0.78        60
     neutral       0.67      0.44      0.53        18
    negative       0.77      0.87      0.82        71

    accuracy                           0.77       149
   macro avg       0.75      0.69      0.71       149
weighted avg       0.77      0.77      0.77       149



## Step 8. Save the trained model and download it

In [17]:
import shutil
model.save_pretrained("sentiment_distilbert_v3")
tok.save_pretrained("sentiment_distilbert_v3")
shutil.make_archive("sentiment_distilbert_v3", "zip", ".", "sentiment_distilbert_v3")
from google.colab import files
files.download("sentiment_distilbert_v3.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
from google.colab import files
up = files.upload()
print(list(up.keys()))

Saving train_aspects.csv to train_aspects (2).csv
['train_aspects (2).csv']


In [19]:
!mv -f "train_aspects (2).csv" train_aspects.csv
print(len(pd.read_csv("train_aspects.csv")))

719


In [20]:
camp = pd.read_csv("train_aspects.csv")
camp = camp[camp["sentiment"].isin(LABELS)].copy()
camp["aspect_term"] = camp["aspect_term"].fillna("general").astype(str)
camp["text"] = camp["text"].astype(str)
print("campus training rows:", len(camp), camp["sentiment"].value_counts().to_dict())

camp_ds = DS(camp)
args2 = TrainingArguments(
    output_dir="out2", num_train_epochs=3, per_device_train_batch_size=16,
    learning_rate=1e-5, weight_decay=0.01, fp16=True, save_strategy="no",
    logging_steps=20, report_to="none", seed=42,
)
trainer2 = Trainer(model=model, args=args2, train_dataset=camp_ds)
trainer2.train()

campus training rows: 719 {'negative': 383, 'positive': 248, 'neutral': 88}


Step,Training Loss
20,0.130089
40,0.121343
60,0.108509
80,0.053002
100,0.059481
120,0.058633


TrainOutput(global_step=135, training_loss=0.08874866874129683, metrics={'train_runtime': 4.7572, 'train_samples_per_second': 453.416, 'train_steps_per_second': 28.378, 'total_flos': 16184337815106.0, 'train_loss': 0.08874866874129683, 'epoch': 3.0})

In [21]:
val_df = val.reset_index(drop=True)
pred = trainer2.predict(val_ds).predictions.argmax(axis=1)
val_df["pred"] = [LABELS[i] for i in pred]
print(val_df[(val_df.sentiment=="neutral") & (val_df.pred!="neutral")][["text","aspect_term","pred"]].to_string())

                                               text aspect_term      pred
3                 too much waiting for just a maggi       maggi  negative
34                mess food is okay nothing special        food  negative
135           queue at lunch moves at a normal pace       queue  negative
139    librarian is polite enough, doesnt talk much   librarian  positive
143  lab staff answer doubts when we ask, thats all   lab staff  negative


In [22]:
pred = trainer2.predict(val_ds).predictions.argmax(axis=1)
gold = np.array(val_ds.y)
print("macro-F1:", round(f1_score(gold, pred, average="macro", labels=[0,1,2], zero_division=0), 3))
print("accuracy:", round(accuracy_score(gold, pred), 3))
print(confusion_matrix(gold, pred, labels=[0,1,2]), " rows=gold, cols=pred", LABELS)
print(classification_report(gold, pred, labels=[0,1,2], target_names=LABELS, zero_division=0))

macro-F1: 0.829
accuracy: 0.866
[[48  4  8]
 [ 1 13  4]
 [ 2  1 68]]  rows=gold, cols=pred ['positive', 'neutral', 'negative']
              precision    recall  f1-score   support

    positive       0.94      0.80      0.86        60
     neutral       0.72      0.72      0.72        18
    negative       0.85      0.96      0.90        71

    accuracy                           0.87       149
   macro avg       0.84      0.83      0.83       149
weighted avg       0.87      0.87      0.86       149



In [ ]:
import shutil
model.save_pretrained("sentiment_distilbert_v2")
tok.save_pretrained("sentiment_distilbert_v2")
shutil.make_archive("sentiment_distilbert_v2", "zip", ".", "sentiment_distilbert_v2")
files.download("sentiment_distilbert_v2.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
!pip install -q transformers accelerate

In [3]:
from google.colab import files
up = files.upload()   # choose sentiment_distilbert_v2.zip and the NEW validation_aspects.csv

Saving validation_aspects.csv to validation_aspects.csv
Saving sentiment_distilbert_v2.zip to sentiment_distilbert_v2.zip


In [4]:
!unzip -q sentiment_distilbert_v2.zip
import pandas as pd, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

LABELS = ["positive", "neutral", "negative"]; L2I = {l: i for i, l in enumerate(LABELS)}
tok = AutoTokenizer.from_pretrained("sentiment_distilbert_v2")
model = AutoModelForSequenceClassification.from_pretrained("sentiment_distilbert_v2")

class DS(torch.utils.data.Dataset):
    def __init__(self, df):
        self.enc = tok(list(df["aspect_term"]), list(df["text"]), truncation=True, max_length=128, padding=True)
        self.y = [L2I[s] for s in df["sentiment"]]
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.y[i])
        return item

def score(model, csv):
    d = pd.read_csv(csv); d = d[d["sentiment"].isin(LABELS)].copy()
    d["aspect_term"] = d["aspect_term"].fillna("general").astype(str); d["text"] = d["text"].astype(str)
    ds = DS(d)
    t = Trainer(model=model, args=TrainingArguments(output_dir="tmp", report_to="none"))
    pred = t.predict(ds).predictions.argmax(axis=1); gold = np.array(ds.y)
    print("macro-F1:", round(f1_score(gold, pred, average="macro", labels=[0,1,2], zero_division=0), 3),
          " accuracy:", round(accuracy_score(gold, pred), 3))
    print(confusion_matrix(gold, pred, labels=[0,1,2]), " rows=true, cols=predicted", LABELS)
    print(classification_report(gold, pred, labels=[0,1,2], target_names=LABELS, zero_division=0))

score(model, "validation_aspects.csv")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

macro-F1: 0.574  accuracy: 0.812
[[50  0 10]
 [ 9  0  9]
 [ 0  0 71]]  rows=true, cols=predicted ['positive', 'neutral', 'negative']
              precision    recall  f1-score   support

    positive       0.85      0.83      0.84        60
     neutral       0.00      0.00      0.00        18
    negative       0.79      1.00      0.88        71

    accuracy                           0.81       149
   macro avg       0.55      0.61      0.57       149
weighted avg       0.72      0.81      0.76       149

